In [1]:
import numpy as np
from collections import Counter

class Node:
    """Clase para representar un nodo del árbol de decisión[cite: 51]."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        # Inicializa os atributos do nodo [cite: 53]
        self.feature = feature       # Índice de la característica para dividir
        self.threshold = threshold   # Valor umbral para la división
        self.left = left             # Hijo izquierdo (Node)
        self.right = right           # Hijo derecho (Node)
        self.value = value           # Valor de la clase (solo para nodos hoja)
        
    def is_leaf(self):
        # Comproba se o nodo e unha folla (non ten fillos) [cite: 54, 55]
        return self.value is not None


class DecisionTree:
    """Clase para construir e utilizar o modelo de árbore de decisión[cite: 29]."""
    def __init__(self, max_depth=None, min_samples_split=2):
        # Inicializa os hiperparametros da arbore [cite: 30]
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        
    def fit(self, X, y):
        # Construe a arbore de decision a partir dos datos de adestramento [cite: 46, 47, 48]
        # Aseguramos que X e y sean arrays de numpy
        X = np.array(X)
        y = np.array(y)
        self.root = self._build_tree(X, y)
        
    def _build_tree(self, X, y, depth=0):
        # Funcion recursiva que construe a arbore de decision [cite: 33, 34]
        num_samples, num_features = X.shape
        num_labels = len(np.unique(y))
        
        # Criterio de parada
        if self.is_finished(depth, num_samples, num_labels):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
            
        # Encontrar la mejor división
        best_feat, best_thresh = self.best_split(X, y, num_features)
        
        # Si no se encuentra una división válida (ej. ganancia de info = 0)
        if best_feat is None:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
            
        # Crear los subárboles de forma recursiva
        left_idxs, right_idxs = self._create_split(X[:, best_feat], best_thresh)
        left = self._build_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._build_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(best_feat, best_thresh, left, right)

    def is_finished(self, depth, num_samples, num_labels):
        # Comproba se se alcanzou a profundidade maxima ou se o
        # numero de mostras e menor que o minimo para dividir [cite: 31, 32]
        if (self.max_depth is not None and depth >= self.max_depth) or \
           num_samples < self.min_samples_split or \
           num_labels == 1:
            return True
        return False

    def best_split(self, X, y, num_features):
        # Busca a mellor division entre as caracteristicas disponibles [cite: 42, 43]
        best_gain = -1
        split_idx, split_thresh = None, None
        
        for feat_idx in range(num_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thr in thresholds:
                gain = self.information_gain(y, X_column, thr)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thr
                    
        return split_idx, split_thresh

    def information_gain(self, y, X_column, threshold):
        # Calcula a ganancia de informacion dunha division [cite: 37, 38]
        # Entropía original
        parent_entropy = self.entropy(y)
        
        # Generar partición
        left_idxs, right_idxs = self._create_split(X_column, threshold)
        
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0
            
        # Calcular media ponderada de la entropía de los hijos
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self.entropy(y[left_idxs]), self.entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r
        
        # Calcular ganancia
        ig = parent_entropy - child_entropy
        return ig

    def _create_split(self, X_column, threshold):
        # Crea a division dos datos en funcion dunha caracteristica e un umbral [cite: 39, 40, 41]
        # Devuelve los índices de las filas que van a la izquierda y a la derecha
        left_idxs = np.argwhere(X_column <= threshold).flatten()
        right_idxs = np.argwhere(X_column > threshold).flatten()
        return left_idxs, right_idxs

    def entropy(self, y):
        # Calcula a entropia dun conxunto de etiquetas [cite: 35, 36]
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])

    def predict(self, X):
        # Predice as etiquetas das mostras de entrada [cite: 49, 50]
        X = np.array(X)
        return np.array([self.traverse_tree(x, self.root) for x in X])

    def traverse_tree(self, x, node):
        # Funcion recursiva para predecir a etiqueta dunha mostra seguindo a arbore [cite: 44, 45]
        if node.is_leaf():
            return node.value
            
        if x[node.feature] <= node.threshold:
            return self.traverse_tree(x, node.left)
        return self.traverse_tree(x, node.right)
        
    def _most_common_label(self, y):
        """Función auxiliar para encontrar la clase mayoritaria en una hoja."""
        counter = Counter(y)
        value = counter.most_common(1)[0][0]
        return value